In [ ]:
%reload_ext autoreload
%autoreload 2

from IPython.display import display, HTML
display(HTML("<style>.container { width:100% !important; }</style>"))

import os
import logging
import yaml

import pandas as pd
import numpy as np

import ferdinand.utils as utl   
import ferdinand.sqlite_utils as sqlutl
import ferdinand.clustering_utils as clutl

from pathlib import Path
from types import SimpleNamespace


from sklearn.neighbors import kneighbors_graph
import igraph as ig
import leidenalg as la

import matplotlib.pyplot as plt
from sklearn.decomposition import PCA
import umap


# Purpose and Documentation

## Purpose

This notebook uses <b>Leiden clustering</b> to automatically group preprocessed radiographs by visual similarity, then enables manual validation and curation to exclude non-whole-body X-rays and low-quality images. The curated and validated datasets produced here serve as input for downstream ML pipeline stages.

## Location 

_&lt;project_dir&gt;/notebooks/310_curate_dataset.ipynb_

## Prerequisites

- A valid runtime configuration file (_../config/config.yml_) with correctly defined _centers_, _run_name_, _parameters_, and _directories_
- Successful execution of the template setup cells from _NNN_notebook_template.ipynb_
- Existing _../data/_ and _../reports/_ directories 
- Existing SQLite database ../data/&lt;run_name&gt;/&lt;all_data_dir&gt;/IMPC_radiographs.db
- Image files have already been preprocessed and are present in _../data/&lt;run_name&gt;/&lt;preprocessed_dir&gt;_

## Workflow

- Extract image features (CNN-based, using SWIN or CLIP models)
- Build k-nearest-neighbor graphs per imaging center
- Apply Leiden community detection to identify clusters of visually similar images
- Visualize clusters using PCA and UMAP
- Generate interactive HTML galleries per cluster for manual review
- Mark clusters/images for exclusion based on QC criteria
- Update SQLite database with cluster assignments and curation decisions


# Definition / Settings

This section initializes the notebook environment: 
- loads the runtime configuration from _../config/config.yml_
- creates the required directory structure
- sets up logging and the SQLite connection

**Do not proceed past this section** </br>
Missing paths and an uninitialized database connection will cause subsequent cells to crash.

## General settings

In [ ]:
# load the configuration file
config_path = Path('../config/config.yml')
with open(config_path, 'r') as file:
    config = yaml.safe_load(file)

print(pd.DataFrame(config['settings'].items(), columns=['configuration parameter', 'set to']).to_markdown(tablefmt="pretty", index=False))

## Set up runtime paths

In [ ]:
# create directories as per configuration
dirs = SimpleNamespace(**config['directories'])
for key, value in vars(dirs).items():
    dirs.__dict__[key] = os.path.join(config['project_root'], config['data_dir'], config['run_name'], value)
    # os.makedirs(dirs.__dict__[key], exist_ok=True)
    if os.path.exists(dirs.__dict__[key]):
        print(f" Directory created: {dirs.__dict__[key]}")
    else:
        print(f" ERROR: Directory does not exist: {dirs.__dict__[key]}")

## Store runtime configuration when necessary

In [ ]:
# copy the current '../config/config.yml' to '../reports/'
dest_filename = f"config_{config['run_name']}.yml"
dest_path = os.path.join(config['project_root'], config['reports_dir'], config['run_name'], dest_filename)
if not os.path.exists(dest_path): 
    commmand_string = f"cp {config_path} {dest_path}"
    os.system(commmand_string)

    if os.path.exists(dest_path): 
        print(f" Current runtime configuration \'{config_path}' saved to \'{dest_path}\'")
    else:
        print(f" ERROR: could not save configuration file to \'{dest_path}\'")

## Set up logger

In [ ]:
# setup logging
notebook_file = sorted(list(Path.cwd().glob('*.ipynb')), key=lambda p: p.stat().st_birthtime, reverse=True)[0].stem 

log_path = os.path.join(config['project_root'], config['reports_dir'], config['run_name'], 'logs')
os.makedirs(log_path, exist_ok=True)

log_file = os.path.join(log_path, f"{notebook_file}.log")
print(f"Log file: {log_file}")

try:
    logger = utl.setup_logger(log_file, log_level=logging.INFO)
    print("Logger successfully set up.")
except Exception as e:
    print(f"Error setting up logger: {e}")

## Set up SQLite connection

In [ ]:
# expected SQLlite database file
sqlite_file = os.path.join(dirs.all_data_dir, config['sqlite_db'])

# if already exists, connect to it
if os.path.exists(sqlite_file):
    conn = sqlutl.connect_db(sqlite_file)
    if conn is not None:
        print(f"connected to database {sqlite_file}\n")
else: 
    print(f" ERROR: No SQLite database found!")

# Build _Leiden_ clusters per center

This section extracts image features (CNN embeddings), builds k-nearest-neighbor graphs, applies Leiden community detection, and visualizes the resulting clusters.

## Workflow

- **Extract features** (optional): Use SWIN or CLIP model to compute embeddings (1024-dim or 512-dim vectors) for all preprocessed images. Save as _.npy_ files.
- **Load features**: Reuse pre-computed features from previous runs if available (much faster).
- **Build KNN graph**: For each center, construct a k-nearest-neighbor connectivity matrix, then convert to igraph object.
- **Apply _Leiden_**: Search over resolution parameters to find optimal cluster count and modularity. Save cluster labels.
- **Visualize**: Create 2D projections (PCA, UMAP) colored by cluster ID to inspect separation quality.
- **Generate galleries**: Create interactive HTML files showing thumbnail images per cluster for manual review.

## Key parameters
- **n_neighbors**: Number of nearest neighbors to connect in KNN graph (typically 3–300 depending on dataset size).
- **resolution**: _Leiden_ algorithm parameter controlling cluster granularity (0.1–2.0). Lower = fewer clusters.
- **seed**: Random seed for reproducibility (set to 42).

**Feature extraction is computationally expensive (~1–10 hours for 10k+ images). Always save and reuse features rather than re-computing.**

In [ ]:
center_paths = {}
center_features = {}
center_labels = {}

In [ ]:
clusters_dir = os.path.join(config['project_root'], config['reports_dir'], config['run_name'], "clusters")
os.makedirs(clusters_dir, exist_ok=True)
if os.path.exists(clusters_dir):
    print(f" Directory created: {clusters_dir}")
else:
    print(f" ERROR: Directory does not exist: {clusters_dir}")

## Extract features

**Feature extraction is slow**: Extracting features for 10k+ images can take hours. Pre-compute once and reuse.

In [ ]:
my_centers = config['centers']
print(f"Centers to be processed: {my_centers}")

In [ ]:
for center in my_centers:

    # select images 
    image_files = utl.select_jpegs_by_column(conn=conn, 
                                             db_table='impc_data',
                                             column='center',
                                             value=center, 
                                             source_path=dirs.preprocessed_dir)

    if len(image_files) > 0:
        # extract features using microsoft/swin-tiny-patch4-window7-224
        center_paths[center], center_features[center] = clutl.extract_features(image_files=image_files)

        np.save(os.path.join(clusters_dir, f"{center}_SWIN_features.npy"), center_features[center])
        np.save(os.path.join(clusters_dir, f"{center}_SWIN_paths.npy"), center_paths[center])
    else:
        print(f"No files found for center: {center}")

## Load existing features

In [ ]:
for center in my_centers:
    
    features_file = os.path.join(clusters_dir, f"{center}_SWIN_features.npy")
    paths_file = os.path.join(clusters_dir, f"{center}_SWIN_paths.npy")
    if os.path.exists(features_file) and os.path.exists(paths_file):
        center_features[center] = np.load(features_file)
        center_paths[center] = np.load(paths_file)

    print(f"{center} features size: {center_features[center].size if center in center_features.keys() else '-'}")

## Build graph and apply _Leiden_

In [ ]:
for center in my_centers:
    if center in center_features:
        results = clutl.find_best_resolution(center_features[center])
        resolution = np.round(results['best_resolution'], 1)
        print(f"Center: {center} - Best resolution: {resolution}")
    
        A = kneighbors_graph(center_features[center], n_neighbors=5, mode='connectivity', include_self=False)
        A = A.maximum(A.T)
        sources, targets = A.nonzero()
        # g = ig.Graph(zip(sources, targets), directed=False)

        g = ig.Graph(
            n=A.shape[0],
            edges=np.column_stack((sources, targets)),
            directed=False
        )

        partition = la.find_partition(g, la.RBConfigurationVertexPartition, seed=42, n_iterations=-1, 
                                      resolution_parameter=resolution)
        print(partition.summary())
        center_labels[center] = partition.membership

In [ ]:
for center in my_centers:
    if center in center_labels:
        df = pd.DataFrame({
            "image_path": center_paths[center],
            "cluster": center_labels[center]
        })
        df.to_csv(os.path.join(clusters_dir, f"{center}_SWIN_leiden_clusters_nbrs_5_res_{resolution}.csv"), index=False)
        # print(df.head())
        # print(df.cluster.unique())

## Load existing clusters

## PCA

In [ ]:
for center in center_features.keys():
    pca = PCA(n_components=2)
    reduced = pca.fit_transform(center_features[center])

    plt.scatter(reduced[:,0], reduced[:,1], c=center_labels[center], s=10)
    plt.title(f"{center}: Leiden Clustering (Liste von JPEGs)")
    plt.show()

## UMAP

In [ ]:
for center in center_features.keys():

    # UMAP reduction
    reducer = umap.UMAP(n_components=2, random_state=42)
    reduced = reducer.fit_transform(center_features[center])

    # Visualisation 
    plt.figure(figsize=(6, 5))
    plt.scatter(reduced[:, 0], reduced[:, 1], c=center_labels[center], s=10, cmap="tab20")
    plt.title(f"{center}: Leiden Clustering (UMAP, JPEG Features)")
    plt.xlabel("UMAP-1")
    plt.ylabel("UMAP-2")
    plt.show()

## Create a thumbnail gallery for each cluster

# Curate dataset based on clusters

## Workflow

## Create thumbnail gallery